In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# סימולציה מדויקת עם primitives של Qiskit SDK

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בחדשות יותר.

```
qiskit[all]~=2.3.0
```
</details>
ה-primitives הייחוסיים ב-Qiskit SDK מבצעים סימולציות statevector מקומיות. סימולציות אלו אינן תומכות
במידול רעש של המכשיר, אך הן שימושיות לאב-טיפוס מהיר של אלגוריתמים לפני שמתבוננים בטכניקות סימולציה מתקדמות יותר ([שימוש ב-Qiskit Aer](./simulate-stabilizer-circuits)) או בהרצה על מכשירים אמיתיים ([Qiskit Runtime primitives](primitives)).

ה-Estimator Primitive יכול לחשב ערכי ציפייה של Circuit, וה-Sampler Primitive יכול לדגום
מהתפלגויות הפלט של Circuit.

הסעיפים הבאים מראים כיצד להשתמש ב-primitives הייחוסיים להרצת הזרימה שלך באופן מקומי.

## שימוש ב-Estimator הייחוסי
המימוש הייחוסי של `EstimatorV2` ב-`qiskit.primitives` שרץ על סימולטורי statevector מקומיים
הוא המחלקה [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). היא יכולה לקבל Circuit, observables ופרמטרים כקלטים ומחזירה את ערכי הציפייה המחושבים באופן מקומי.

הקוד הבא מכין את הקלטים שישמשו בדוגמאות שיבואו. סוג הקלט הצפוי עבור
ה-observables הוא [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). שים לב
ש-Circuit בדוגמה הוא פרמטרי, אך ניתן גם להריץ את ה-Estimator על Circuit לא-פרמטריים.

> **Note:** כל Circuit שמועבר ל-Estimator חייב **שלא** לכלול **מדידות**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** זרימת העבודה של Qiskit Runtime primitives דורשת ש-Circuit וה-observables יומרו לשימוש רק בהוראות הנתמכות על ידי ה-QPU (המכונות מעגלי *instruction set architecture (ISA)* ו-observables). ה-primitives הייחוסיים עדיין מקבלים הוראות מופשטות, מכיוון שהם מסתמכים על סימולציות statevector מקומיות, אך Transpile של ה-Circuit עשוי עדיין להיות מועיל מבחינת אופטימיזציית ה-Circuit.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### אתחול Estimator
צור מופע של [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### הרצה וקבלת תוצאות
דוגמה זו משתמשת רק ב-Circuit אחד (מסוג [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) וב-observable אחד.

הרץ את ההערכה על ידי קריאה למתודה [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run), שמחזירה מופע של אובייקט [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). ניתן לקבל את התוצאות מהמשימה (כאובייקט [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
עם המתודה [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### קבלת ערך הציפייה מהתוצאה
תוצאות ה-primitives מפיקות מערך של אובייקטי [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult), כאשר כל פריט במערך הוא אובייקט `PubResult` שמכיל בנתוניו את מערך ההערכות המתאים לכל שילוב Circuit-observable ב-PUB.

כדי לאחזר את ערכי הציפייה והמטא-נתונים עבור הערכת ה-Circuit הראשונה (ובמקרה זה, היחידה), עלינו לגשת ל-[`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) של ההערכה עבור PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Primitives examples](primitives-examples#estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run the Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### הגדרת אפשרויות הרצה של Estimator
כברירת מחדל, ה-Estimator הייחוסי מבצע חישוב statevector מדויק המבוסס על
המחלקה [`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
אולם, ניתן לשנות זאת כדי להכניס את ההשפעה של תקורת הדגימה (המכונה גם "shot noise").

ה-Estimator מקבל ארגומנט `precision` שמבטא את פסי השגיאה שמימוש ה-primitive
אמור לשאוף אליהם עבור הערכות ערכי ציפייה. זוהי תקורת הדגימה ומוגדרת אך ורק במתודה `.run()`. זה מאפשר לכוון את האפשרות עד לרמת ה-PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

לדוגמה מלאה, ראה את הדף [דוגמאות Primitives](primitives-examples#estimator-examples).

## שימוש ב-Sampler הייחוסי
המימוש הייחוסי של `SamplerV2` ב-`qiskit.primitives` הוא המחלקה [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). היא מקבלת Circuit ופרמטרים כקלטים ומחזירה את התוצאות מדגימה מהתפלגויות ההסתברות של מצבי הפלט כהתפלגות קוואזי-הסתברות של מצבי פלט.

הקוד הבא מכין את הקלטים שישמשו בדוגמאות שיבואו. שים לב
שדוגמאות אלו מריצות Circuit פרמטרי בודד, אך ניתן גם להריץ את ה-Sampler
על Circuit לא-פרמטריים.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** כל Circuit קוונטי שמועבר ל-Sampler **חייב** לכלול מדידות.

> **Tip:** זרימת העבודה של Qiskit Runtime primitives דורשת ש-Circuit יומרו לשימוש רק בהוראות הנתמכות על ידי ה-QPU (המכונות ISA circuits). ה-primitives הייחוסיים עדיין מקבלים הוראות מופשטות, מכיוון שהם מסתמכים על סימולציות statevector מקומיות, אך Transpile של ה-Circuit עשוי עדיין להיות מועיל מבחינת אופטימיזציית ה-Circuit.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### אתחול `SamplerV2`
צור מופע של [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### הרצה וקבלת תוצאות

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 515, '00': 509}


Primitives מקבלים מספר PUBs כקלטים, וכל PUB מקבל תוצאה משלו. לפיכך, ניתן להריץ Circuit שונים עם שילובי פרמטר/observable שונים, ולאחזר את תוצאות ה-PUB:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Primitives examples](./primitives-examples#sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>